# Social Media Emotion Analyzer - Data Preprocessing & Feature Extraction
## Member 1: Data Processing Pipeline

**Complete production code for Google Colab**

In [2]:
!pip install -q gensim scikit-learn nltk pandas numpy scipy matplotlib seaborn

In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

import pickle
import scipy.sparse as sp
from google.colab import files

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Step 1: Upload Dataset

In [4]:
print("Upload your dataset files (training.csv, test.csv, validation.csv)")
uploaded = files.upload()
print(f"✓ Uploaded files: {list(uploaded.keys())}")

Upload your dataset files (training.csv, test.csv, validation.csv)


Saving test.csv to test (1).csv
Saving training.csv to training (1).csv
Saving validation.csv to validation (1).csv
✓ Uploaded files: ['test (1).csv', 'training (1).csv', 'validation (1).csv']


## Step 2: Load Data

In [5]:
train_df = pd.read_csv('training.csv')
test_df = pd.read_csv('test.csv')
val_df = pd.read_csv('validation.csv')

emotion_map = {0: 'Sadness', 1: 'Joy', 2: 'Love', 3: 'Anger', 4: 'Fear', 5: 'Surprise'}
train_df['emotion'] = train_df['label'].map(emotion_map)
test_df['emotion'] = test_df['label'].map(emotion_map)
val_df['emotion'] = val_df['label'].map(emotion_map)

print(f"✓ Training: {len(train_df)} samples")
print(f"✓ Test: {len(test_df)} samples")
print(f"✓ Validation: {len(val_df)} samples")

✓ Training: 16000 samples
✓ Test: 2000 samples
✓ Validation: 2000 samples


## Step 3: Text Preprocessing

In [6]:
class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[@#]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def preprocess(self, text):
        text = self.clean_text(text)
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t not in self.stop_words and len(t) > 2]
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        return ' '.join(tokens)

preprocessor = TextPreprocessor()

print("Preprocessing training data...", end='', flush=True)
train_df['text_cleaned'] = train_df['text'].apply(lambda x: preprocessor.preprocess(x))
print(" ✓")

print("Preprocessing test data...", end='', flush=True)
test_df['text_cleaned'] = test_df['text'].apply(lambda x: preprocessor.preprocess(x))
print(" ✓")

print("Preprocessing validation data...", end='', flush=True)
val_df['text_cleaned'] = val_df['text'].apply(lambda x: preprocessor.preprocess(x))
print(" ✓")

Preprocessing training data... ✓
Preprocessing test data... ✓
Preprocessing validation data... ✓


## Step 4: Save Cleaned Data

In [7]:
train_df.to_csv('train_cleaned.csv', index=False)
test_df.to_csv('test_cleaned.csv', index=False)
val_df.to_csv('val_cleaned.csv', index=False)

print("✓ Cleaned CSV files saved")
print(f"  - train_cleaned.csv: {len(train_df)} rows")
print(f"  - test_cleaned.csv: {len(test_df)} rows")
print(f"  - val_cleaned.csv: {len(val_df)} rows")

✓ Cleaned CSV files saved
  - train_cleaned.csv: 16000 rows
  - test_cleaned.csv: 2000 rows
  - val_cleaned.csv: 2000 rows


## Step 5: TF-IDF Feature Extraction

In [8]:
print("Extracting TF-IDF features...")

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['text_cleaned'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['text_cleaned'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['text_cleaned'])

print(f"✓ TF-IDF Features Extracted")
print(f"  - Training: {X_train_tfidf.shape}")
print(f"  - Test: {X_test_tfidf.shape}")
print(f"  - Validation: {X_val_tfidf.shape}")

sp.save_npz('X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('X_test_tfidf.npz', X_test_tfidf)
sp.save_npz('X_val_tfidf.npz', X_val_tfidf)
print("✓ TF-IDF features saved")

Extracting TF-IDF features...
✓ TF-IDF Features Extracted
  - Training: (16000, 5000)
  - Test: (2000, 5000)
  - Validation: (2000, 5000)
✓ TF-IDF features saved


## Step 6: Word2Vec Feature Extraction

In [9]:
print("Extracting Word2Vec features...")

sentences_train = [text.split() for text in train_df['text_cleaned']]
sentences_test = [text.split() for text in test_df['text_cleaned']]
sentences_val = [text.split() for text in val_df['text_cleaned']]

print("Training Word2Vec model...", end='', flush=True)
w2v_model = Word2Vec(
    sentences=sentences_train,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)
print(" ✓")
print(f"  - Vocabulary: {len(w2v_model.wv)} words")

def sentences_to_w2v_vectors(sentences, model, vector_size=300):
    vectors = []
    for sentence in sentences:
        word_vectors = [model.wv[word] for word in sentence if word in model.wv]
        if len(word_vectors) > 0:
            vector = np.mean(word_vectors, axis=0)
        else:
            vector = np.zeros(vector_size)
        vectors.append(vector)
    return np.array(vectors)

print("Creating Word2Vec vectors...", end='', flush=True)
X_train_w2v = sentences_to_w2v_vectors(sentences_train, w2v_model)
X_test_w2v = sentences_to_w2v_vectors(sentences_test, w2v_model)
X_val_w2v = sentences_to_w2v_vectors(sentences_val, w2v_model)
print(" ✓")

print(f"✓ Word2Vec Features Created")
print(f"  - Training: {X_train_w2v.shape}")
print(f"  - Test: {X_test_w2v.shape}")
print(f"  - Validation: {X_val_w2v.shape}")

np.save('X_train_w2v.npy', X_train_w2v)
np.save('X_test_w2v.npy', X_test_w2v)
np.save('X_val_w2v.npy', X_val_w2v)
print("✓ Word2Vec features saved")

Extracting Word2Vec features...
Training Word2Vec model... ✓
  - Vocabulary: 6544 words
Creating Word2Vec vectors... ✓
✓ Word2Vec Features Created
  - Training: (16000, 300)
  - Test: (2000, 300)
  - Validation: (2000, 300)
✓ Word2Vec features saved


## Step 7: Save Labels

In [10]:
y_train = train_df['label'].values
y_test = test_df['label'].values
y_val = val_df['label'].values

np.save('y_train.npy', y_train)
np.save('y_test.npy', y_test)
np.save('y_val.npy', y_val)

print("✓ Labels saved")
print(f"  - y_train: {y_train.shape}")
print(f"  - y_test: {y_test.shape}")
print(f"  - y_val: {y_val.shape}")

✓ Labels saved
  - y_train: (16000,)
  - y_test: (2000,)
  - y_val: (2000,)


## Step 8: Save Models & Tools

In [11]:
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("✓ TF-IDF vectorizer saved")

w2v_model.save('word2vec_model.bin')
print("✓ Word2Vec model saved")

with open('text_preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("✓ Text preprocessor saved")

feature_names_tfidf = np.array(tfidf_vectorizer.get_feature_names_out())
metadata = {
    'emotion_map': emotion_map,
    'feature_names_tfidf': feature_names_tfidf,
    'w2v_vocab_size': len(w2v_model.wv),
    'w2v_vector_size': w2v_model.vector_size,
    'tfidf_features': X_train_tfidf.shape[1],
}

with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("✓ Metadata saved")

✓ TF-IDF vectorizer saved
✓ Word2Vec model saved
✓ Text preprocessor saved
✓ Metadata saved


## Step 9: Download All Files

In [12]:
import os
import zipfile

output_files = [
    'train_cleaned.csv', 'test_cleaned.csv', 'val_cleaned.csv',
    'X_train_tfidf.npz', 'X_test_tfidf.npz', 'X_val_tfidf.npz',
    'X_train_w2v.npy', 'X_test_w2v.npy', 'X_val_w2v.npy',
    'y_train.npy', 'y_test.npy', 'y_val.npy',
    'tfidf_vectorizer.pkl', 'word2vec_model.bin', 'text_preprocessor.pkl', 'metadata.pkl'
]

print("Creating zip file...")
with zipfile.ZipFile('emotion_analyzer_data.zip', 'w') as zipf:
    for file in output_files:
        if os.path.exists(file):
            zipf.write(file)
            print(f"  ✓ Added {file}")

print("\n✓ Zip file created: emotion_analyzer_data.zip")
print("\nDownloading all files...")
files.download('emotion_analyzer_data.zip')
for file in output_files:
    if os.path.exists(file):
        files.download(file)

Creating zip file...
  ✓ Added train_cleaned.csv
  ✓ Added test_cleaned.csv
  ✓ Added val_cleaned.csv
  ✓ Added X_train_tfidf.npz
  ✓ Added X_test_tfidf.npz
  ✓ Added X_val_tfidf.npz
  ✓ Added X_train_w2v.npy
  ✓ Added X_test_w2v.npy
  ✓ Added X_val_w2v.npy
  ✓ Added y_train.npy
  ✓ Added y_test.npy
  ✓ Added y_val.npy
  ✓ Added tfidf_vectorizer.pkl
  ✓ Added word2vec_model.bin
  ✓ Added text_preprocessor.pkl
  ✓ Added metadata.pkl

✓ Zip file created: emotion_analyzer_data.zip



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 10: Summary

In [13]:
print("="*70)
print("DATA PROCESSING COMPLETE!")
print("="*70)
print(f"\nDataset:")
print(f"  Training: {len(train_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Validation: {len(val_df)} samples")

print(f"\nFeature Extraction:")
print(f"  TF-IDF: {X_train_tfidf.shape[1]} features")
print(f"  Word2Vec: 300 dimensions")
print(f"  Vocabulary: {len(w2v_model.wv)} words")

print(f"\nEmotions:")
for emotion_id, emotion_name in emotion_map.items():
    count = (train_df['label'] == emotion_id).sum()
    pct = (count / len(train_df)) * 100
    print(f"  {emotion_name}: {count} ({pct:.1f}%)")

print("\n✓ All files ready for download!")
print("="*70)

DATA PROCESSING COMPLETE!

Dataset:
  Training: 16000 samples
  Test: 2000 samples
  Validation: 2000 samples

Feature Extraction:
  TF-IDF: 5000 features
  Word2Vec: 300 dimensions
  Vocabulary: 6544 words

Emotions:
  Sadness: 4666 (29.2%)
  Joy: 5362 (33.5%)
  Love: 1304 (8.2%)
  Anger: 2159 (13.5%)
  Fear: 1937 (12.1%)
  Surprise: 572 (3.6%)

✓ All files ready for download!
